# 01 — NBA Player Stats Retrieval
### NBA.com API | 2016-17 through 2025-26

This notebook gets advanced player stats from the NBA API for 10 seasons (2016-17 through 2025-26). It combines all regular season data into one CSV file.

**Source:** NBA.com Official Stats via `nba_api` (`LeagueDashPlayerStats`, Advanced measure type)

**Output:** `nba_player_stats.csv`

**Sections:** Setup → Helper Functions → Fetch Targets → Data Extraction → Clean & Preprocess → Export

**Dependencies:** `nba_api`, `pandas`, `re`, `time`

In [1]:
%pip install --upgrade nba_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 7.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import pandas as pd
import re
import unicodedata
from nba_api.stats.endpoints import leaguedashplayerstats

### Section 1 — Setup & Helper Functions

`season_start_year`: Checks the season format and gets the start year.
`make_seasons`: Creates a list of all seasons in the correct format.
`fetch_advanced_stats`: Requests stats from the NBA API and adds SEASON and SEASON_TYPE columns.

In [5]:
# target seasons
start_season = "2016-17"
end_season = "2025-26"


def season_start_year(season):
    # Target season validation.
    # The function takes a season in the 2020-21 format and returns the start year as an integer.
    if not re.fullmatch(r"\d{4}-\d{2}", season):
        raise ValueError("Enter seasons in the 2020-21 format.")
    return int(season[:4])


def make_seasons(start_season, end_season):
    # Return value is a list of seasons in the '2020-21' format.
    start_year = season_start_year(start_season)
    end_year = season_start_year(end_season)

    if end_year < start_year:
        raise ValueError("The end season must be the same as or later than the start season.")

    return [f"{str(year)}-{str(year + 1)[-2:]}" for year in range(start_year, end_year + 1)]


def fetch_advanced_stats(season, season_type):
    # Stats retrieval from NBA API.
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star=season_type,
        measure_type_detailed_defense="Advanced",
        timeout=60,
    ).get_data_frames()[0]

    stats["SEASON"] = season
    stats["SEASON_TYPE"] = season_type
    return stats

### Section 2 — Generation of Fetch Targets

We create a list of target seasons and define the season type (Regular Season). We also set a sleep time between requests to avoid API errors.

In [7]:
# Season & Season Type Setting
seasons = make_seasons(start_season, end_season)
season_types = ["Regular Season"]


request_sleep_seconds = 3
print("Seasons to fetch:", seasons)
print("Season types:", season_types)
print("Sleep between requests:", request_sleep_seconds, "seconds")

Seasons to fetch: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Season types: ['Regular Season']
Sleep between requests: 3 seconds


### Section 3 — Data Extraction Loop with Rate Limiting

We use a loop to request data for each season and season type.
The NBA API has strict request limits, so we use these steps to get data safely:
1. Sleep for 3 seconds between requests.
2. Set a 60-second timeout for the API connection.
3. Use try-except to catch errors. If a request fails, the loop continues and saves the error to a list.
4. Combine all successful data into one DataFrame at the end. 

In [9]:
# Other basic setting for data fetching
all_frames = []
errors = []

# Fetch data
for season in seasons:
    for season_type in season_types:
        try:
            all_frames.append(fetch_advanced_stats(season, season_type))

        except Exception as exc:
            error_text = str(exc)
            errors.append({"SEASON": season, "SEASON_TYPE": season_type, "ERROR": error_text})
            print(f"Failed to fetch: {season} / {season_type}: {error_text}")

        time.sleep(request_sleep_seconds)

all_stats = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

print("Rows fetched:", len(all_stats))
if errors:
    print("Some requests failed:")
    display(pd.DataFrame(errors))

Rows fetched: 5492


In [11]:
print(f'Shape of the fetched stats data: {all_stats.shape}')
display(all_stats.sample(5))

Shape of the fetched stats data: (5492, 81)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT,SEASON,SEASON_TYPE
4151,1628370,Malik Monk,Malik,1610612758,SAC,26.0,72,42,30,0.583,...,196,149,80,69,93,85,314,1,2023-24,Regular Season
936,203816,Scotty Hopson,Scotty,1610612742,DAL,28.0,1,0,1,0.000,...,5,530,529,530,529,511,529,1,2017-18,Regular Season
2251,1630267,Facundo Campazzo,Facundo,1610612743,DEN,30.0,65,43,22,0.662,...,451,358,279,239,371,322,447,1,2020-21,Regular Season
3959,1630535,Greg Brown III,Greg,1610612742,DAL,22.0,6,3,3,0.500,...,7,508,521,530,491,500,272,1,2023-24,Regular Season
190,2037,Jamal Crawford,Jamal,1610612746,LAC,37.0,82,51,31,0.622,...,196,281,87,63,113,94,326,1,2016-17,Regular Season


### Section 4 — Data Cleaning & Preprocessing

We review the `all_stats` DataFrame to confirm that player statistics have been successfully fetched. We also set `TEAM_ABBREVIATION` to `TOT` for any players that have played for multiple teams in a single season to match with the Basketball Reference convention. Then, we construct a unique `id` feature to allow for clean data joins with the player salary data, drop unnecessary columns, and filter out players averaging fewer than 10 minutes per game to exclude unreliable per-possession statistics. Finally, we set `id` as the DataFrame index.

In [13]:
SUFFIXES = {'jr', 'jr.', 'sr', 'sr.', 'ii', 'iii', 'iv', 'v'}

TEAM_ABBREV_MAP = {
    'BKN': 'BRK',
    'CHA': 'CHO',
    'PHX': 'PHO',
}

def build_id(name: str, team: str, season: str) -> str:
    # Normalize unicode characters to ASCII (e.g. Jokić -> Jokic)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')

    # Normalize team abbreviation to Basketball Reference convention
    team = TEAM_ABBREV_MAP.get(team.upper(), team.upper())
    
    # Builds a unique player-season identifier in the format 'leb_james_lal_24'
    tokens = name.lower().split()
    tokens = [t for t in tokens if t not in SUFFIXES]
    if not tokens:
        return f'unknown_{team.lower()}_{str(season)[-2:]}'
    first = tokens[0][:4]
    last = tokens[-1]
    return f'{first}_{last}_{team.lower()}_{str(season)[-2:]}'

def clean_preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    # Create a copy of the DataFrame to avoid modifying the original data
    df = df.copy()

    # Overwrite TEAM_ABBREVIATION to TOT if player has played for multiple teams in a single season (TEAM_COUNT > 1)
    df.loc[df['TEAM_COUNT'] > 1, 'TEAM_ABBREVIATION'] = 'TOT'

    # Create a unique identifier for each player-season 
    df['id'] = df.apply(lambda x: build_id(x['PLAYER_NAME'], x['TEAM_ABBREVIATION'], x['SEASON']), axis = 1)

    # Drop any unnecessary columns (un-used ids, team-based statistics, rankings, sp_work_ adjusted stats)
    cols_to_drop = [
        'PLAYER_ID', 'NICKNAME', 'TEAM_ID', 'SEASON_TYPE', 'W', 'L', 'TEAM_COUNT',
        *[col for col in df.columns if col.endswith('_RANK')],
        *[col for col in df.columns if col.startswith('sp_work_')]
    ]
    df = df.drop(columns = cols_to_drop)

    # Retain only the columns needed for analysis
    # This drops any unnecessary columns (un-used ids, team-based statistics, rankings, sp_work_ adjusted stats)
    cols_to_keep = [
        'id', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W_PCT', 'MIN',
        'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'REB_PCT',
        'TS_PCT', 'USG_PCT', 'PIE', 'POSS', 'FGM_PG', 'FGA_PG', 'FG_PCT', 'SEASON'
    ]
    df = df[cols_to_keep]

    # Filter out any players with less than 10 minutes of playing time per game
    # This helps to exclude extreme per-possession stats like usage percent or true shooting percent
    df = df[df['MIN'] >= 10]

    # Sort by player name and season
    df = df.sort_values(by = ['PLAYER_NAME', 'SEASON'])

    # Set index of dataframe to new 'id'
    df = df.set_index('id')

    return df

In [15]:
cleaned_stats = clean_preprocess_data(all_stats)

In [17]:
print(f'Shape of the cleaned stats data: {cleaned_stats.shape}')
display(cleaned_stats.sample(5))

Shape of the cleaned stats data: (4450, 20)


,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,W_PCT,MIN,OFF_RATING,DEF_RATING,NET_RATING,AST_PCT,AST_TO,REB_PCT,TS_PCT,USG_PCT,PIE,POSS,FGM_PG,FGA_PG,FG_PCT,SEASON
id,,,,,,,,,,,,,,,,,,,,
edmo_sumner_ind_20,Edmond Sumner,IND,24.0,31,0.516,14.4,104.7,107.3,-2.6,0.167,2.50,0.049,0.491,0.167,0.058,958,2.0,4.6,0.430,2019-20
dori_finney-smith_dal_21,Dorian Finney-Smith,DAL,28.0,60,0.600,32.0,117.2,112.1,5.1,0.070,2.13,0.084,0.609,0.120,0.074,3935,3.7,7.8,0.472,2020-21
jayl_adams_atl_19,Jaylen Adams,ATL,23.0,34,0.382,12.6,102.5,109.1,-6.5,0.201,2.32,0.063,0.474,0.131,0.055,954,1.1,3.2,0.345,2018-19
just_harper_phi_17,Justin Harper,PHI,27.0,3,0.333,10.4,95.5,113.6,-18.1,0.100,0.50,0.000,0.500,0.208,0.039,67,1.7,4.0,0.417,2016-17
jade_ivey_det_24,Jaden Ivey,DET,22.0,77,0.182,28.8,109.9,117.7,-7.7,0.196,1.56,0.059,0.536,0.244,0.076,4706,5.4,12.6,0.429,2023-24


### Section 5 — CSV Export & Data Validation

We save the final data to a CSV file. The notebook checks if the data is empty. If it is correct, it prints the total rows, columns, and previews the first 5 rows.

In [19]:
# Export to CSV
if cleaned_stats.empty:
    print("No data to export.")
else:
    output_file = "./data/nba_player_stats.csv"
    cleaned_stats.to_csv(output_file, encoding="utf-8")

    print("CSV export complete:", output_file)
    print("Rows:", len(cleaned_stats))
    print("Columns:", len(cleaned_stats.columns))
    display(cleaned_stats.sample(5))

CSV export complete: ./data/nba_player_stats.csv
Rows: 4450
Columns: 20


,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,W_PCT,MIN,OFF_RATING,DEF_RATING,NET_RATING,AST_PCT,AST_TO,REB_PCT,TS_PCT,USG_PCT,PIE,POSS,FGM_PG,FGA_PG,FG_PCT,SEASON
id,,,,,,,,,,,,,,,,,,,,
keld_johnson_sas_24,Keldon Johnson,SAS,24.0,69,0.246,29.5,108.5,116.1,-7.7,0.142,1.95,0.088,0.565,0.212,0.098,4399,5.7,12.5,0.454,2023-24
naji_marshall_nop_23,Naji Marshall,NOP,25.0,77,0.519,23.3,111.2,111.4,-0.1,0.147,1.95,0.079,0.539,0.172,0.086,3779,3.2,7.4,0.433,2022-23
dema_derozan_chi_22,DeMar DeRozan,CHI,32.0,76,0.566,36.1,113.1,112.0,1.1,0.228,2.07,0.072,0.590,0.308,0.148,5723,10.2,20.2,0.504,2021-22
john_williams_lal_19,Johnathan Williams,LAL,24.0,24,0.417,15.5,110.0,102.4,7.6,0.051,0.81,0.123,0.599,0.158,0.078,790,2.7,4.6,0.591,2018-19
andr_bogut_tot_17,Andrew Bogut,TOT,32.0,27,0.296,21.6,95.1,97.4,-2.2,0.133,1.14,0.189,0.460,0.104,0.092,1131,1.4,3.0,0.469,2016-17


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fa7bfbce-9eb0-4b44-99aa-9c5ae30d8022' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>